# Dynamic tool selection in LangChain

A simple tutorial using a shop assistant. By the end, you will understand how to change the tools a model sees at runtime, using `create_agent` and `@wrap_model_call`.

The selection rule and checks run without an API key. The optional agent examples use Groq, already used in this repository. All tool results are fictional local data.

## 1. What does dynamic tool selection mean?

With a static tool list, the model sees the same tools on every call. With dynamic selection, your application chooses a relevant subset **before each model call**. The model then decides whether to call one of those tools and supplies its arguments.

These are two separate decisions: the application selects what is available; the model selects what to use. A prompt saying 'only use shipping tools' does not change the tool list sent to the model. Middleware does.

Our application provides a mode for each run:

| Mode | Tools shown to the model | Example question |
| --- | --- | --- |
| `catalog` | `lookup_product` | How much is the mug? |
| `shipping` | `lookup_delivery` | When will order A100 arrive? |
| Anything else | None | Explain that lookup is unavailable |

```text
Question + run context
        -> middleware selects tools
        -> model chooses a tool (or answers directly)
        -> LangChain executes the tool
        -> middleware selects tools again
        -> model gives the final answer
```

A smaller tool list can reduce irrelevant tool descriptions and make tool choice easier. Here, the mode stays fixed during a run; in other applications, selection can also depend on changing agent state.

## 2. Setup

Use Python 3.11+ and a Jupyter kernel from this project's `.venv`. This tutorial uses LangChain 1.x. If the imports below fail, uncomment and run the installation line, then restart the kernel. `%pip` installs into the current kernel's environment.

The example was checked locally with `langchain==1.4.3`, `langchain-core==1.6.3`, and `langchain-groq==1.1.3`.

In [ ]:
# Uncomment if these packages are missing from your notebook environment.
# %pip install "langchain>=1.4.3,<2" "langchain-core>=1.6.3,<2" "langchain-groq>=1.1.3,<2"

In [ ]:
from dataclasses import dataclass
from typing import Callable

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import tool
from langchain_core.messages import AIMessage, ToolMessage

## 3. Define two tools

A tool is a Python function the agent can call. `@tool` exposes its name, docstring, and argument schema to the model. These tools use dictionaries so we can focus on selection rather than external services.

In [ ]:
@tool
def lookup_product(product: str) -> str:
    """Look up the price and stock of a product: mug or notebook."""
    catalog = {
        'mug': 'Mug: INR 350; 12 in stock.',
        'notebook': 'Notebook: INR 120; 30 in stock.',
    }
    return catalog.get(product.strip().lower(), 'Product not found in the demo catalog.')


@tool
def lookup_delivery(order_id: str) -> str:
    """Look up the delivery estimate for an order: A100 or A101."""
    deliveries = {
        'A100': 'Order A100: shipped; expected in 2 business days.',
        'A101': 'Order A101: packing; expected in 5 business days.',
    }
    return deliveries.get(order_id.strip().upper(), 'Order not found in the demo records.')


ALL_TOOLS = [lookup_product, lookup_delivery]
print('Registered tools:', [t.name for t in ALL_TOOLS])
print(lookup_product.invoke({'product': 'mug'}))
print(lookup_delivery.invoke({'order_id': 'A100'}))

## 4. Write the selection rule

`RunContext` holds application-supplied configuration for one invocation. The mode is passed through `context=`, not inferred from the user's message. A dataclass gives that context a clear shape.

The selection function is ordinary Python. Unknown modes return an empty list, making the fallback easy to see.

In [ ]:
@dataclass
class RunContext:
    mode: str = 'catalog'


TOOLS_BY_MODE = {
    'catalog': {'lookup_product'},
    'shipping': {'lookup_delivery'},
}


def select_tools(mode: str, available_tools: list) -> list:
    allowed_names = TOOLS_BY_MODE.get(mode, set())
    return [t for t in available_tools if t.name in allowed_names]


for mode in ['catalog', 'shipping', 'unknown']:
    selected = select_tools(mode, ALL_TOOLS)
    print(f'{mode:8} -> {[t.name for t in selected]}')

## 5. Apply the rule before each model call

`@wrap_model_call` lets us intercept a model request. The three important pieces are:

1. `request.runtime.context` reads the configuration passed to this run.
2. `request.override(tools=selected)` creates a request with the filtered tools.
3. `handler(...)` continues the model call with that request.

The print statement shows exactly which tools the model is offered. Filtering `request.tools` also preserves any earlier narrowing by other middleware.

In [ ]:
@wrap_model_call
def dynamic_tool_selection(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    mode = request.runtime.context.mode
    selected = select_tools(mode, request.tools)
    print(f'[mode={mode}] Tools offered: {[t.name for t in selected]}')
    return handler(request.override(tools=selected))

## 6. Check the rule without an API key

These checks confirm the exact lists we intend to expose. They test the selection policy; the next section connects it to an actual agent.

In [ ]:
assert [t.name for t in select_tools('catalog', ALL_TOOLS)] == ['lookup_product']
assert [t.name for t in select_tools('shipping', ALL_TOOLS)] == ['lookup_delivery']
assert select_tools('unknown', ALL_TOOLS) == []
assert select_tools('shipping', [lookup_product]) == []
assert len(ALL_TOOLS) == 2  # Selection leaves the registry intact.
print('All selection checks passed.')

## 7. Connect the middleware to an agent (optional live example)

Set `RUN_LIVE = True` to run the remaining agent examples. You need a Groq API key and internet access; model requests consume API resources. The key is read from `GROQ_API_KEY` or requested with a hidden prompt. It is never printed or saved by this notebook.

Choose a tool-calling model available to your account. The default below is `llama-3.3-70b-versatile`; override it with the `GROQ_MODEL` environment variable if needed. Model availability changes: consult [Groq's model list](https://console.groq.com/docs/models).

Register **both tools** with `create_agent` so LangChain can execute either one. The middleware filters which of those registered tools is sent to the model on each call. We create the agent once and reuse it with different contexts.

In [ ]:
import getpass
import os

RUN_LIVE = False
agent = None

if RUN_LIVE:
    from langchain_groq import ChatGroq

    api_key = os.environ.get('GROQ_API_KEY') or getpass.getpass('Groq API key: ')
    if not api_key.strip():
        raise ValueError('A Groq API key is required for the live examples.')
    model = ChatGroq(
        model=os.environ.get('GROQ_MODEL', 'llama-3.3-70b-versatile'),
        api_key=api_key,
        temperature=0,
        timeout=30,
        max_retries=1,
    )
    del api_key
    agent = create_agent(
        model=model,
        tools=ALL_TOOLS,
        middleware=[dynamic_tool_selection],
        context_schema=RunContext,
        system_prompt=(
            'You are a shop assistant using fictional demo data. '
            'For product prices, stock, or delivery estimates, use an available '
            'lookup tool. If the appropriate lookup tool is unavailable, say '
            'you cannot look up that information in this mode. Do not invent facts. '
            'After a successful lookup, give a concise answer using its result.'
        ),
    )
else:
    print('Live examples disabled. Set RUN_LIVE = True and rerun this cell to enable them.')

In [ ]:
def run_demo(question: str, mode: str):
    if agent is None:
        print(f'Skipped live example: {mode} / {question}')
        return None

    print(f'Question: {question}')
    result = agent.invoke(
        {'messages': [{'role': 'user', 'content': question}]},
        context=RunContext(mode=mode),
        config={'recursion_limit': 10},
    )
    for message in result['messages']:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                print(f"Tool called: {call['name']} with {call['args']}")
        elif isinstance(message, ToolMessage):
            print(f'Tool result: {message.content}')
    print('Answer:', result['messages'][-1].content)
    return result

### Catalog mode

Expect the middleware to offer only `lookup_product`. The model should call it with `product='mug'` and answer using INR 350 and 12 in stock. You will usually see the middleware log twice: before the tool call and before the final answer. The wording and number of calls can vary with the model.

In [ ]:
catalog_result = run_demo('How much is the mug, and is it in stock?', 'catalog')

### Shipping mode

The same agent now sees only `lookup_delivery`. Expect a lookup for `A100`, followed by an answer saying the order has shipped and is expected in 2 business days.

In [ ]:
shipping_result = run_demo('When will order A100 arrive?', 'shipping')

### Same question, different available tools

Ask the shipping question in catalog mode. The model sees `lookup_product` but receives no delivery tool schema. With the system instructions above, it should explain that delivery lookup is unavailable. The middleware controls tool visibility; the model's response wording remains probabilistic.

The second example uses an unknown mode, so the model is offered no tools at all.

In [ ]:
mismatch_result = run_demo('When will order A100 arrive?', 'catalog')
unknown_result = run_demo('How much is the mug?', 'unknown')

## 8. What to remember

`tools=ALL_TOOLS` registers the tools the agent can execute. `request.override(tools=selected)` narrows the tools offered for one model call. `context=RunContext(...)` supplies the selection input for this invocation. No agent rebuild is needed.

This example filters tools registered in advance. Adding newly discovered tools at runtime is a separate pattern. Also, hiding a tool schema is not a complete authorization boundary: enforce real permissions when tools execute and in the underlying service.

### Try it yourself

1. Add a `combined` mode exposing both tools, then ask for the mug's price and order A100's delivery estimate in one question.
2. Add a `lookup_return_policy` tool. Register it in `ALL_TOOLS` and expose it in a new `returns` mode.
3. Consider which application signal should drive selection in your own project: workflow stage, user role, or task category.

### Troubleshooting

If imports fail, install the packages from section 2 and restart the kernel. If live cells say 'Skipped', enable `RUN_LIVE` and rerun the setup cell. If Groq reports an authentication or model error, check your key or select a tool-calling model available to your account.

### References

- [LangChain: dynamically selecting tools](https://docs.langchain.com/oss/python/langchain/middleware/custom#dynamically-selecting-tools) — the middleware pattern used here.
- [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents) — `create_agent` and run context.
- [ChatGroq integration](https://docs.langchain.com/oss/python/integrations/chat/groq) — provider setup.